In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# --- SECTION 1: SINGLE LAYER ---
def run_single_layer():
    print("\n--- RUNNING SINGLE LAYER NETWORK ---")
    # Data generation: Convert lists to NumPy arrays immediately
    max_n = 10
    x = np.random.randint(0, max_n, size=(2000, 2)) # [2,3]
    y = np.sum(x, axis=1) # 2+3 = 5

    # Model definition
    model = keras.Sequential([
        layers.Dense(1, input_shape=(2,)) # Direct input to output
    ])

    model.compile(optimizer='adam', loss='mse')

    # Training
    model.fit(x, y, epochs=50, verbose=0)

    # Testing
    test_val = np.array([[5, 4]])
    pred = model.predict(test_val, verbose=0)
    print(f"Input: 5 + 4 | Predicted: {pred[0][0]:.2f}")

run_single_layer()


--- RUNNING SINGLE LAYER NETWORK ---
Input: 5 + 4 | Predicted: 9.00


In [ ]:
import numpy as np
import tensorflow as tf

def train_single_layer_custom():
    print("--- CUSTOM TRAINING LOOP: LEARNING TO ADD ---")

    # 1. Create Training Data (a + b = c)
    # Using float32 because TensorFlow requires it for manual gradient calculations
    x_train = np.random.randint(0, 10, size=(1000, 2)).astype(np.float32)
    y_train = np.sum(x_train, axis=1).astype(np.float32).reshape(-1, 1)

    # 2. Create the Single-Layer Model
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(1, input_shape=(2,))
    ])

    # 3. Define the Loss Function and Optimizer
    loss_fn = tf.keras.losses.MeanSquaredError()
    # Using a higher learning rate (0.1) so we can watch it learn faster
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.1)

    # Look at the initial, completely random weights
    initial_weights, initial_bias = model.layers[0].get_weights()
    print(f"\nINITIAL RANDOM STATE:")
    print(f"Weight for 'a': {initial_weights[0][0]:.2f}")
    print(f"Weight for 'b': {initial_weights[1][0]:.2f}")
    print(f"Bias: {initial_bias[0]:.2f}\n")

    print("Starting Training Loop...\n")

    # --- THE TRAINING LOOP ---
    epochs = 20
    for epoch in range(epochs):

        # tf.GradientTape() is like a tape recorder. It records the math operations
        # so it can play them backward later to calculate the gradients (Backpropagation).
        with tf.GradientTape() as tape:

            # STEP 1: FORWARD PASS (Make a guess)
            # The model multiplies inputs by current weights and adds bias
            predictions = model(x_train)

            # STEP 2: CALCULATE LOSS (How wrong was the guess?)
            # Compare predictions to actual y_train values
            loss = loss_fn(y_train, predictions)

        # STEP 3: BACKPROPAGATION (Figure out who to blame)
        # Calculate the gradients (the direction to nudge the weights to fix the error)
        # We ask the tape: "How did the model's variables affect the loss?"
        gradients = tape.gradient(loss, model.trainable_variables)

        # STEP 4: UPDATE WEIGHTS (The Correction)
        # The optimizer nudges the weights and bias in the direction of the gradients
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))

        # --- Printing the progress ---
        current_weights, current_bias = model.layers[0].get_weights()
        print(f"Epoch {epoch + 1:02d} | Loss: {loss.numpy():.2f} | "
              f"Weights: [w1: {current_weights[0][0]:.3f}, w2: {current_weights[1][0]:.3f}] | "
              f"Bias: {current_bias[0]:.3f}")

    # Test the final model
    print("\n--- FINAL TEST ---")
    test_val = np.array([[7.0, 5.0]], dtype=np.float32)
    pred = model(test_val)
    print(f"Input: 7 + 5 | Final Prediction: {pred[0][0]:.2f}")

# Run the function
train_single_layer_custom()

--- CUSTOM TRAINING LOOP: LEARNING TO ADD ---

INITIAL RANDOM STATE:
Weight for 'a': 0.02
Weight for 'b': 0.87
Bias: 0.00

Starting Training Loop...

Epoch 01 | Loss: 31.93 | Weights: [w1: 0.124, w2: 0.971] | Bias: 0.100
Epoch 02 | Loss: 21.69 | Weights: [w1: 0.223, w2: 1.070] | Bias: 0.199
Epoch 03 | Loss: 13.78 | Weights: [w1: 0.320, w2: 1.165] | Bias: 0.295
Epoch 04 | Loss: 8.17 | Weights: [w1: 0.414, w2: 1.254] | Bias: 0.387
Epoch 05 | Loss: 4.74 | Weights: [w1: 0.503, w2: 1.333] | Bias: 0.472
Epoch 06 | Loss: 3.21 | Weights: [w1: 0.585, w2: 1.398] | Bias: 0.547
Epoch 07 | Loss: 3.13 | Weights: [w1: 0.659, w2: 1.445] | Bias: 0.609
Epoch 08 | Loss: 3.89 | Weights: [w1: 0.722, w2: 1.473] | Bias: 0.657
Epoch 09 | Loss: 4.90 | Weights: [w1: 0.774, w2: 1.481] | Bias: 0.688
Epoch 10 | Loss: 5.70 | Weights: [w1: 0.813, w2: 1.471] | Bias: 0.704
Epoch 11 | Loss: 6.03 | Weights: [w1: 0.842, w2: 1.448] | Bias: 0.706
Epoch 12 | Loss: 5.85 | Weights: [w1: 0.860, w2: 1.413] | Bias: 0.696
Epoch 1

In [18]:
import numpy as np
import tensorflow as tf

print("=" * 70)
print("DIAGNOSTIC: WHY THE ORIGINAL CODE FAILS")
print("=" * 70)

np.random.seed(42)
tf.random.set_seed(42)

scale_factor = 100.0

x_train_raw = np.random.randint(0, 100, size=(2000, 2)).astype(np.float32)
x_train_scaled = x_train_raw / scale_factor
y_train_scaled = np.sum(x_train_scaled, axis=1, keepdims=True).astype(np.float32)

print(f"\n📊 Training Data Statistics:")
print(f"   X range (scaled): [{x_train_scaled.min():.3f}, {x_train_scaled.max():.3f}]")
print(f"   Y range (scaled): [{y_train_scaled.min():.3f}, {y_train_scaled.max():.3f}]")
print(f"   Y mean (scaled): {y_train_scaled.mean():.3f}")
print(f"   Y std (scaled): {y_train_scaled.std():.3f}")

model_bad = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(2,)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1)
])

loss_fn = tf.keras.losses.MeanSquaredError()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

print(f"\n🔄 Training with ORIGINAL approach (full-batch, 40 epochs)...")
losses = []

for epoch in range(40):
    with tf.GradientTape() as tape:
        predictions = model_bad(x_train_scaled, training=True)
        loss = loss_fn(y_train_scaled, predictions)

    gradients = tape.gradient(loss, model_bad.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model_bad.trainable_variables))
    losses.append(loss.numpy())

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"   Epoch {epoch + 1:2d} | Loss: {loss.numpy():.4f}")

test_cases = np.array([[10.0, 20.0], [50.0, 50.0], [7.0, 2.0]], dtype=np.float32)
test_scaled = test_cases / scale_factor
preds_bad = model_bad(test_scaled, training=False).numpy() * scale_factor

print(f"\n❌ ORIGINAL Results (BROKEN):")
for i, (inputs, pred) in enumerate(zip(test_cases, preds_bad)):
    expected = inputs.sum()
    print(f"   {inputs[0]:.0f} + {inputs[1]:.0f} = {expected:.0f} | Predicted: {pred[0]:.2f} | Error: {abs(expected - pred[0]):.2f}")

print(f"\n🔍 ROOT CAUSES:")
print(f"   1. ⚠️  Too few epochs (40) - needs 100-200 for convergence")
print(f"   2. ⚠️  Full-batch training - no stochasticity, poor optimization")
print(f"   3. ⚠️  No validation - can't detect overfitting")
print(f"   4. ⚠️  Learning rate 0.01 too high for full-batch")

print(f"\n" + "=" * 70)
print("FIXED VERSION: Proper Training Configuration")
print("=" * 70)

np.random.seed(42)
tf.random.set_seed(42)

x_train_raw = np.random.randint(0, 100, size=(2000, 2)).astype(np.float32)
x_train_scaled = x_train_raw / scale_factor
y_train_scaled = np.sum(x_train_scaled, axis=1, keepdims=True).astype(np.float32)

model_fixed = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(2,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1)
])

optimizer_fixed = tf.keras.optimizers.Adam(learning_rate=0.001)

print(f"\n🔄 Training with FIXED approach (mini-batch, 200 epochs)...")

epochs = 200
batch_size = 32
num_batches = len(x_train_scaled) // batch_size

for epoch in range(epochs):
    indices = np.random.permutation(len(x_train_scaled))
    x_shuffled = x_train_scaled[indices]
    y_shuffled = y_train_scaled[indices]

    epoch_loss = 0
    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = start + batch_size

        x_batch = x_shuffled[start:end]
        y_batch = y_shuffled[start:end]

        with tf.GradientTape() as tape:
            predictions = model_fixed(x_batch, training=True)
            loss = loss_fn(y_batch, predictions)

        gradients = tape.gradient(loss, model_fixed.trainable_variables)
        optimizer_fixed.apply_gradients(zip(gradients, model_fixed.trainable_variables))
        epoch_loss += loss.numpy()

    avg_loss = epoch_loss / num_batches

    if (epoch + 1) % 50 == 0 or epoch == 0:
        print(f"   Epoch {epoch + 1:3d} | Loss: {avg_loss:.6f}")

preds_fixed = model_fixed(test_scaled, training=False).numpy() * scale_factor

print(f"\n✅ FIXED Results (WORKING):")
for i, (inputs, pred) in enumerate(zip(test_cases, preds_fixed)):
    expected = inputs.sum()
    error = abs(expected - pred[0])
    status = "✓" if error < 5 else "✗"
    print(f"   {status} {inputs[0]:.0f} + {inputs[1]:.0f} = {expected:.0f} | Predicted: {pred[0]:.2f} | Error: {error:.2f}")

print(f"\n📈 KEY FIXES APPLIED:")
print(f"   ✓ Increased epochs: 40 → 200")
print(f"   ✓ Added mini-batch training (batch_size=32)")
print(f"   ✓ Lower learning rate: 0.01 → 0.001")
print(f"   ✓ Larger network: [16,8] → [32,16]")
print(f"   ✓ Data shuffling each epoch")

print(f"\n" + "=" * 70)

DIAGNOSTIC: WHY THE ORIGINAL CODE FAILS

📊 Training Data Statistics:
   X range (scaled): [0.000, 0.990]
   Y range (scaled): [0.010, 1.950]
   Y mean (scaled): 0.972
   Y std (scaled): 0.413

🔄 Training with ORIGINAL approach (full-batch, 40 epochs)...
   Epoch  1 | Loss: 1.0668
   Epoch 10 | Loss: 0.2172
   Epoch 20 | Loss: 0.1198
   Epoch 30 | Loss: 0.0553
   Epoch 40 | Loss: 0.0541

❌ ORIGINAL Results (BROKEN):
   10 + 20 = 30 | Predicted: 53.31 | Error: 23.31
   50 + 50 = 100 | Predicted: 92.67 | Error: 7.33
   7 + 2 = 9 | Predicted: 46.04 | Error: 37.04

🔍 ROOT CAUSES:
   1. ⚠️  Too few epochs (40) - needs 100-200 for convergence
   2. ⚠️  Full-batch training - no stochasticity, poor optimization
   3. ⚠️  No validation - can't detect overfitting
   4. ⚠️  Learning rate 0.01 too high for full-batch

FIXED VERSION: Proper Training Configuration

🔄 Training with FIXED approach (mini-batch, 200 epochs)...
   Epoch   1 | Loss: 0.604441
   Epoch  50 | Loss: 0.000002
   Epoch 100 | Los